## 01 — Read SCADA


In [ ]:
import pandas as pd, time, os
from openpyxl import load_workbook
SRC="/mnt/user-data/uploads/SCADA__monitoring_dataset_2020.xlsx"
t=time.time()
wb=load_workbook(SRC, read_only=True, data_only=True)
for n in [n for n in wb.sheetnames if n.endswith("_data.csv")]:
    tid=n[:4]
    if os.path.exists(f"raw_{tid}.pkl"): continue
    it=wb[n].iter_rows(values_only=True)
    hdr=next(it)
    df=pd.DataFrame(list(it), columns=hdr)
    df.to_pickle(f"raw_{tid}.pkl")
    print(tid, df.shape, round(time.time()-t), "s", flush=True)


## 02 — Merge & Check


In [ ]:
import pandas as pd, numpy as np
frames=[]
for i in range(1,11):
    tid=f"WT{i:02d}"
    df=pd.read_pickle(f"raw_{tid}.pkl")
    df.insert(0,"Turbine_ID",tid)
    frames.append(df)
d=pd.concat(frames,ignore_index=True)
print("merged shape:",d.shape)
d["Timestamp"]=pd.to_datetime(d["Timestamp"],errors="coerce")
print("bad timestamps:",int(d["Timestamp"].isna().sum()))
d=d.sort_values(["Turbine_ID","Timestamp"]).reset_index(drop=True)
print("range:",d["Timestamp"].min(),"->",d["Timestamp"].max())
print("rows per turbine:\n",d.groupby("Turbine_ID").size().to_string())

# gaps
print("\n--- gaps (step != 10min) ---")
d["_dt"]=d.groupby("Turbine_ID")["Timestamp"].diff()
g=d[d["_dt"]>pd.Timedelta("10min")]
print("gap count:",len(g))
print(g.groupby("Turbine_ID")["_dt"].agg(["count","max"]).to_string())
print("top gaps:\n",g.nlargest(8,"_dt")[["Turbine_ID","Timestamp","_dt"]].to_string())
print("step<10min or zero:",int((d["_dt"]<pd.Timedelta("10min")).sum()))
d=d.drop(columns="_dt")

# duplicates
dup=d.duplicated(["Turbine_ID","Timestamp"],keep=False)
print("\n--- duplicates (Turbine_ID+Timestamp) rows:",int(dup.sum()))
print("fully identical duplicate rows:",int(d.duplicated(keep="first").sum()))

# missing_data flag
print("\n--- missing_data flag ---")
print(d["missing_data"].value_counts(dropna=False).to_string())
print(d.groupby("Turbine_ID")["missing_data"].sum().to_string())

# NaN
na=d.isna().sum()
na=na[na>0].sort_values(ascending=False)
print("\n--- columns with NaN:",len(na))
print((na/len(d)*100).round(2).head(25).to_string())
print("rows with any NaN:",int(d.isna().any(axis=1).sum()))
print("rows all-NaN (except id/time/flag):",int(d.drop(columns=["Turbine_ID","Timestamp","missing_data"]).isna().all(axis=1).sum()))
d.to_pickle("merged.pkl")


## 03 — Feature Selection


In [ ]:
import pandas as pd, numpy as np, re
d=pd.read_pickle("merged.pkl")
d=d[~d["missing_data"].astype(bool)].copy()
meta=["Turbine_ID","Timestamp"]
cols=[c for c in d.columns if c not in meta]
num=d[cols].apply(pd.to_numeric,errors="coerce")
print("non-numeric cols after coercion NaN increase:",[c for c in cols if num[c].isna().sum()>d[c].isna().sum()])
reason={}
# 1) forbidden
for c in cols:
    if c=="missing_data" or c.startswith("System Logs") or c.startswith("First Alarm") or "AlarmActive" in c:
        reason[c]="forbidden (alarm/log/flag)"
# 2) cumulative / hour counters
for c in cols:
    if c in reason: continue
    if ("hour" in c.lower() and "counter" in c.lower()) or c.endswith("hours [h]") or c.startswith("HourCounters") \
       or "Total accumulated" in c or c in ("Total Active power [W]","Total reactive power [var]"):
        reason[c]="cumulative counter"
# 3) constant / near constant
for c in cols:
    if c in reason: continue
    s=num[c]
    if s.nunique(dropna=True)<=1: reason[c]="constant"
    elif s.value_counts(normalize=True,dropna=True).iloc[0]>0.99: reason[c]="near-constant (>99% same value)"
# 4) high correlation
rem=[c for c in cols if c not in reason]
corr=num[rem].corr().abs()
upper=corr.where(np.triu(np.ones(corr.shape),1).astype(bool))
pairs=[]
dropped=set()
# prefer to keep the Avg column in a correlated group
order=sorted(rem,key=lambda c:(0 if "Avg" in c else 1,c))
keep=[]
for c in order:
    hit=[k for k in keep if corr.loc[c,k]>0.95]
    if hit:
        reason[c]=f"corr>0.95 with '{hit[0]}'"
    else: keep.append(c)
keep=[c for c in rem if c in keep]
print("total cols:",len(cols))
from collections import Counter
print(Counter(v.split(" with")[0].split(" (")[0] for v in reason.values()))
print("kept:",len(keep))
for c in keep: print("  ",c)
pd.DataFrame({"feature":keep}).to_csv("/mnt/user-data/outputs/features_final.csv",index=False)
pd.DataFrame([{"column":c,"reason":r} for c,r in reason.items()]).to_csv("/mnt/user-data/outputs/features_excluded.csv",index=False)


## 04 — Data Preparation


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json, io

INPUT_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
ROOT = INPUT_DIR / 'SCADA_Final_Output'
ROOT.mkdir(exist_ok=True)
clean_candidates = [INPUT_DIR/'scada_cleaned.csv', INPUT_DIR/'scada_cleaned(2).csv']
CLEAN_PATH = next((p for p in clean_candidates if p.exists()), clean_candidates[0])
LOGS_PATH = INPUT_DIR/'logs_all_events.csv.gz'
if not CLEAN_PATH.exists() or not LOGS_PATH.exists():
    from google.colab import files
    print('Upload the missing cleaned SCADA and/or Logs file.')
    uploaded = files.upload()
    CLEAN_PATH = next((p for p in clean_candidates if p.exists()), clean_candidates[0])
assert CLEAN_PATH.exists(), 'Set CLEAN_PATH to your original cleaned CSV.'
assert LOGS_PATH.exists(), 'Set LOGS_PATH to your original Logs CSV.GZ.'
print('SCADA:', CLEAN_PATH.name)
print('Logs:', LOGS_PATH.name)

In [ ]:
FEATURES = ['Generator RPM Avg. [RPM]', 'Generator RPM StdDev [RPM]', 'Generator Bearing Temp. Avg. [°C]', 'Generator Phase1 Temp. Avg. [°C]', 'Generator SlipRing Temp. Avg. [°C]', 'Generator Bearing2 Temp. Avg. [°C]', 'Generator CoolingWater Temp. Avg. [°C]', 'Hydraulic Oil Temp. Avg. [°C]', 'Gear Oil TemperatureLevel1 Avg. [°C]', 'Gear Bearing TemperatureHSGeneratorEnd Avg. [°C]', 'Gear Bearing TemperatureHollowShaftRotor Avg. [°C]', 'Nacelle Temp. Avg. [°C]', 'Avg. direction [°]', 'Ambient WindSpeed Min. [m/s]', 'Ambient WindSpeed Avg. [m/s]', 'Ambient WindSpeed StdDev [m/s]', 'Ambient WindDir Relative Avg. [°]', 'Ambient WindDir Absolute Avg. [°]', 'Ambient Temp. Avg. [°C]', 'Grid InverterPhase1 Temp. Avg. [°C]', 'Grid Production CosPhi Avg.', 'Grid Production VoltagePhase1 Avg. [V]', 'Grid Production CurrentPhase1 Avg. [A]', 'Grid Production Power Min. [W]', 'Grid Busbar Temp. Avg. [°C]', 'Grid Production Power StdDev [W]', 'Grid Production ReactivePower Avg. [W]', 'Grid Production ReactivePower Max. [W]', 'Grid Production ReactivePower Min. [W]', 'Grid Production ReactivePower StdDev [W]', 'Grid Production PossiblePower Max. [W]', 'Grid Production PossiblePower Min. [W]', 'Grid Production PossiblePower StdDev [W]', 'Active power limit [W]', 'Controller Ground Temp. Avg. [°C]', 'Controller Top Temp. Avg. [°C]', 'Controller Hub Temp. Avg. [°C]', 'Controller VCP Temp. Avg. [°C]', 'Controller VCP ChokecoilTemp. Avg. [°C]', 'Controller VCP WaterTemp. Avg. [°C]', 'Blades PitchAngle Avg. [°]', 'Blades PitchAngle StdDev [°]', 'HVTrafo Phase1 Temp. Avg. [°C]', 'HVTrafo AirOutlet Temp. Avg. [°C]', 'Production LatestAverage Active Power Gen 0 Avg. [W]', 'Production LatestAverage Active Power Gen 2 Avg. [W]', 'Production LatestAverage Reactive Power Gen 1 Avg. [var]', 'Production LatestAverage Reactive Power Gen 2 Avg. [var]']
frozen = pd.read_csv(INPUT_DIR / 'SCADA_Group_Split_Assignment.csv')
assert len(FEATURES) == 48 and len(frozen) == 1364


## 05 — Event Labeling


In [ ]:
"""Reproducible, description-based weak labels; not manufacturer ground truth."""
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd


OUT = ROOT / 'corrected_scada'
OUT.mkdir(exist_ok=True)
rules = {}
def assign(codes, category, fault_type, reason):
    for code in codes:
        assert code not in rules, code
        rules[code] = dict(Event_Class=category, Fault_Group=fault_type, Decision_Reason=reason)

# Groups describe the logged symptom/system, not a confirmed root cause.
groups = {
 'Communication':[36,37,49,62,63,305],
 'Controller':[95,97,98,106,237,491,908],
 'Electrical':[126,127,134,315,316,318,328,329,333,336,338,341],
 'Controller_Temperature':[324], 'Electrical_Temperature':[687],
 'Pitch_Hydraulic':[161,602,604,619,620,731,878,909,910,911,912,914,915,916,937,938,939,940,942,806],
 'Lubrication':[501,502,505,807,898],
 'Gear_Oil':[552,844,895,897,862,863,864,865],
 'Generator_Cooling':[546,560],
 'Rotor_Sensor':[639,640], 'Load_Sensor':[723,724,725,818],
 'Sensor_Data':[380,381,412,413], 'UPS_Battery':[852,853],
 'Yaw_Azimuth':[79,355,745], 'Brake_Blade_Lock':[507,508,510,512,607,612],
 'Startup':[907], 'Valve_Signal':[675],
}
for name,codes in groups.items():
    assign(codes,'Fault',name,'Description explicitly reports an error, defect, failed action, lost signal, or abnormal limit in this system. Group denotes logged symptom/system, not root cause.')
assign([100,221,879], 'Fault','Unknown','Description explicitly reports repeated auto-restarts, ERROR, or signal error; a specific fault system cannot be established from this description alone.')
assign([144,356,509,901], 'Environmental','Not_Applicable','Description reports high wind or extreme wind direction; environmental/protection condition, not sufficient evidence of equipment fault.')
assign([239,882], 'Environmental','Not_Applicable','Lightning detection; does not establish equipment damage.')
assign([114,115,225,227,248,265,266,275,276,280,281,282,283,284,310,506,677,678,750,751,805,857,905,926],
       'Operation','Not_Applicable','Description records a measurement, command, connection, operating transition, or completed lubrication action. This event alone neither proves a fault nor proves the entire SCADA row healthy.')
assign([70,228,229,260,261,262,263,264,267,307,360,404,681],
       'Operation','Not_Applicable','Description records reset, configuration, log restoration, or reconnection; no active equipment fault can be inferred from this event alone.')
assign([102,154,156,177,191,222,223,224,226,230,232,233,234,235,236,309,352,353,403,414,415,416,485,493,598,625,685,839,851,858,870,900,920,923,924,925],
       'Unresolved','Unknown','Protection, state, acknowledgement, recovery, abbreviated description, or incompletely explained condition. Do not infer an active component fault or normal operation solely from this event.')

review = (pd.read_csv(LOGS_PATH, low_memory=False)
          .groupby(['Code','Description','Event type','Severity'], dropna=False)
          .size().reset_index(name='Event_Count'))
assert set(review.Code) == set(rules), (set(review.Code)-set(rules),set(rules)-set(review.Code))
records=[]
for code,g in review.groupby('Code',sort=True):
    desc=g.Description.dropna().astype(str).drop_duplicates().tolist()
    records.append(dict(Code=int(code),Description_Examples=' | '.join(desc[:5]),
        Description_Variants=len(desc),Source_Types=' | '.join(sorted(g['Event type'].dropna().unique())),
        Log_Record_Count=int(g.Event_Count.sum()),**rules[code],
        Evidence_Basis='Dataset Description; analyst interpretation, not manufacturer-confirmed'))
rule_table=pd.DataFrame(records)
rule_table.to_csv(OUT/'event_code_rules.csv',index=False,encoding='utf-8-sig')

s=pd.read_csv(CLEAN_PATH,low_memory=False)
original_count=len(s)
legacy=['Condition','Fault_Type','Log_Interval_Fault_Types','Log_Interval_Type_Count',
 'Uncertain_Warning_Interval','SCADA_Code_Fault_Type','Combined_Fault_Types','Combined_Type_Count',
 'Label_Status','Exact_Start_Fault_Types','Proposed_Condition','Proposed_Fault_Type']
s=s.drop(columns=legacy,errors='ignore')
base_cols=s.columns.tolist()
s['Timestamp']=pd.to_datetime(s.Timestamp,format='mixed',errors='raise')
l=pd.read_csv(LOGS_PATH,low_memory=False)
l['Raw_Row_ID']=np.arange(len(l))
l['Start']=pd.to_datetime(l.Detected,format='mixed',errors='coerce')
# Parse numeric Excel serials explicitly, without ambiguous date swapping.
raw_end=l['Reset/Run']
number=pd.to_numeric(raw_end,errors='coerce')
l['End']=pd.to_datetime(raw_end.where(number.isna()),format='mixed',errors='coerce')
numeric=number.notna()
l.loc[numeric,'End']=pd.Timestamp('1899-12-30')+pd.to_timedelta(number[numeric],unit='D')
min_time,max_time=s.Timestamp.min(),s.Timestamp.max()
l['Time_Status']='Start_Only'
has_end=raw_end.notna()
good=l.Start.notna()&l.End.notna()&(l.End>l.Start)
l.loc[good,'Time_Status']='Valid_Interval'
l.loc[has_end&~good,'Time_Status']='Invalid_Interval'
l.loc[l.Start.isna(),'Time_Status']='Invalid_Start'
# A parsed end after the dataset window is not automatically wrong; retain it,
# but explicitly flag numeric out-of-window values for separate treatment.
bad_numeric=numeric&~l.End.between(min_time,max_time)
l.loc[bad_numeric,'Time_Status']='Unresolved_Numeric_End'
l=l.merge(rule_table[['Code','Event_Class','Fault_Group']],on='Code',validate='many_to_one')
# Collapse repeated source records at the same turbine/code/description/start.
# Keep all distinct intervals, not just an arbitrary first end.
events=l.groupby(['Turbine_ID','Code','Description','Start','End','Time_Status','Event_Class','Fault_Group'],dropna=False).agg(
    Source_Types=('Event type',lambda x:' | '.join(sorted(set(x)))),
    Source_Record_Count=('Raw_Row_ID','size')).reset_index()
events['Event_ID']=np.arange(len(events))
# Suppress duplicate start-only records when the same event also has an interval.
key=['Turbine_ID','Code','Description','Start']
interval_keys=set(map(tuple,events.loc[events.Time_Status.eq('Valid_Interval'),key].itertuples(index=False,name=None)))
events['Duplicate_Start_Only']=[st=='Start_Only' and tuple(k) in interval_keys for st,k in zip(events.Time_Status,events[key].itertuples(index=False,name=None))]
events['SCADA_Rows_Linked']=0
events['Link_Method']='Not_Linked'
events.loc[events.Event_Class.eq('Operation'),'Link_Method']='Operational_Record_Not_Used_As_Fault_Or_Normal_Proof'
n=len(s)
faults=[set() for _ in range(n)]
evidence=[set() for _ in range(n)]
blockers=[set() for _ in range(n)]
point_guard=np.zeros(n,dtype=bool)
long_warning=np.zeros(n,dtype=bool)
duration=(events.End-events.Start).dt.total_seconds()
events['Thirty_Day_Warning']=events.Source_Types.str.contains('Warning',regex=False)&duration.eq(30*86400)

for turbine,idx in s.groupby('Turbine_ID').groups.items():
    idx=np.asarray(idx)
    order=np.argsort(s.loc[idx,'Timestamp'].to_numpy())
    idx=idx[order]; times=s.loc[idx,'Timestamp'].to_numpy(dtype='datetime64[ns]')
    for e in events[events.Turbine_ID.eq(turbine)&events.Event_Class.ne('Operation')].itertuples():
        if e.Duplicate_Start_Only or pd.isna(e.Start): continue
        start=np.datetime64(e.Start,'ns')
        if e.Time_Status=='Valid_Interval':
            a=np.searchsorted(times,start,'left'); b=np.searchsorted(times,np.datetime64(e.End,'ns'),'left')
            hit=idx[a:b]; method='Recorded_Interval_Start_Inclusive_End_Exclusive'
        else:
            a=np.searchsorted(times,start,'left')
            hit=idx[a:a+1] if a<len(times) and times[a]==start else np.array([],dtype=int)
            method='Exact_Start_Only'
            # Do not invent a duration or assign fault to a nearest row. Block
            # normal labels at the two 10-minute boundaries around unresolved
            # timing, since aggregation direction is not documented.
            if e.Event_Class!='Operation':
                for boundary in (e.Start.floor('10min'),e.Start.ceil('10min')):
                    j=np.searchsorted(times,np.datetime64(boundary,'ns'))
                    if j<len(times) and times[j]==np.datetime64(boundary,'ns'):
                        point_guard[idx[j]]=True
        events.loc[e.Index,'SCADA_Rows_Linked']=len(hit)
        events.loc[e.Index,'Link_Method']=method if len(hit) else 'No_SCADA_Timestamp_Inside_Recorded_Time'
        for i in hit:
            if e.Event_Class=='Fault':
                faults[i].add(e.Fault_Group)
                evidence[i].add('Logs:'+str(e.Code))
            elif e.Event_Class in ('Environmental','Unresolved'):
                blockers[i].add(e.Event_Class+':'+str(e.Code))
        if e.Thirty_Day_Warning: long_warning[hit]=True

active=pd.to_numeric(s['System Logs First Active Alarm No'],errors='coerce')
active_classes=[]
for i,value in enumerate(active):
    if pd.isna(value):
        active_classes.append('Missing');blockers[i].add('Missing_Active_Code')
    elif value==0:
        active_classes.append('No_Active_Code')
    elif value!=int(value) or int(value) not in rules:
        active_classes.append('Unmapped');blockers[i].add('Unmapped_Active_Code:'+str(value))
    else:
        r=rules[int(value)];active_classes.append(r['Event_Class'])
        if r['Event_Class']=='Fault':
            faults[i].add(r['Fault_Group']);evidence[i].add('SCADA:'+str(int(value)))
        elif r['Event_Class']!='Operation':
            blockers[i].add(r['Event_Class']+':'+str(int(value)))

total=pd.to_numeric(s['HourCounters Average Total Avg. [h]'],errors='coerce')
run=pd.to_numeric(s['HourCounters Average Run Avg. [h]'],errors='coerce')
ok=pd.to_numeric(s['HourCounters Average TurbineOk Avg. [h]'],errors='coerce')
service=pd.to_numeric(s['HourCounters Average ServiceOn Avg. [h]'],errors='coerce')
alarm=pd.to_numeric(s['HourCounters Average AlarmActive Avg. [h]'],errors='coerce')
# Explicit definition of a screened normal-running candidate. Zero alarms alone
# never establishes normal. A known benign code can pass the code screen, but
# a positive AlarmActive counter remains unexplained and blocks normal.
screen=(total.gt(0)&run.eq(total)&ok.eq(total)&service.eq(0)&alarm.eq(0)).to_numpy()
s['Active_Code_Class']=active_classes
s['Operating_Normal_Screen']=screen
s['Recorded_30Day_Warning_Coverage']=long_warning
s['Unresolved_Event_Time_Nearby']=point_guard
s['Fault_Evidence_Types']=[' | '.join(sorted(x)) for x in faults]
s['Fault_Evidence_Codes']=[' | '.join(sorted(x)) for x in evidence]
s['Other_Event_Evidence']=[' | '.join(sorted(x)) for x in blockers]
s['Condition']='Unknown';s['Fault_Type']='Unknown';s['Label_Reason']='Insufficient_Normal_Evidence'
for i,fs in enumerate(faults):
    if fs:
        s.at[i,'Condition']='Fault'
        known=fs-{'Unknown'}
        s.at[i,'Fault_Type']=next(iter(known)) if len(known)==1 and 'Unknown' not in fs else ('Multiple' if len(known)>1 else 'Unknown')
        s.at[i,'Label_Reason']='Explicit_Fault_Evidence'
    elif blockers[i]: s.at[i,'Label_Reason']='Unresolved_Or_Environmental_Event'
    elif point_guard[i]: s.at[i,'Label_Reason']='Event_Time_Unresolved'
    elif screen[i]:
        s.at[i,'Condition']='Normal';s.at[i,'Fault_Type']='No_Fault'
        s.at[i,'Label_Reason']='Screened_Normal_Operation_Candidate'
s['Fault_During_Normal_Operating_Screen']=s.Condition.eq('Fault')&screen
s['Binary_Eligible']=s.Condition.isin(['Normal','Fault'])
s['Type_Eligible']=s.Condition.eq('Fault')&~s.Fault_Type.isin(['Unknown','Multiple'])
s['Label_Basis']='Description_Based_Weak_Label'
assert len(s)==original_count
assert not s[['Condition','Fault_Type','Label_Reason']].isna().any().any()
assert not set(legacy[2:])&set(s.columns)
assert s.loc[s.Condition.eq('Normal'),'Fault_Type'].eq('No_Fault').all()
assert s.loc[s.Condition.eq('Fault'),'Fault_Evidence_Codes'].ne('').all()
assert events.loc[events.Thirty_Day_Warning,'Time_Status'].eq('Valid_Interval').all()
s.to_csv(OUT/'scada_corrected_labels.csv.gz',index=False,compression='gzip')
events.to_csv(OUT/'events_link_audit.csv.gz',index=False,compression='gzip')
events[events.Time_Status.str.contains('Invalid|Unresolved',regex=True)].to_csv(OUT/'event_time_issues.csv',index=False)
counts=s.groupby(['Condition','Fault_Type']).size().rename('SCADA_Row_Count').reset_index()
counts.to_csv(OUT/'label_counts.csv',index=False)
rule_table.groupby('Event_Class').agg(Unique_Codes=('Code','size'),Log_Records=('Log_Record_Count','sum')).to_csv(OUT/'code_class_counts.csv')
events[events.Event_Class.eq('Fault')].groupby('Fault_Group').agg(
    Event_Record_Groups=('Event_ID','size'),Linked_Event_Record_Groups=('SCADA_Rows_Linked',lambda x:int(x.gt(0).sum()))).to_csv(OUT/'fault_event_counts.csv')
summary=dict(rows=len(s),base_columns=len(base_cols),condition_counts=s.Condition.value_counts().to_dict(),
    fault_type_counts=s.loc[s.Condition.eq('Fault'),'Fault_Type'].value_counts().to_dict(),
    thirty_day_events=int(events.Thirty_Day_Warning.sum()),thirty_day_scada_rows=int(long_warning.sum()),
    fault_with_normal_screen=int(s.Fault_During_Normal_Operating_Screen.sum()),
    time_issues=int(events.Time_Status.str.contains('Invalid|Unresolved',regex=True).sum()),
    binary_eligible=int(s.Binary_Eligible.sum()),type_eligible=int(s.Type_Eligible.sum()),
    code_count=len(rule_table),unmatched_fault_event_groups=int(((events.Event_Class=='Fault')&events.SCADA_Rows_Linked.eq(0)&~events.Duplicate_Start_Only).sum()))
(OUT/'summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

assert s.Condition.value_counts().to_dict() == {'Normal':217050,'Fault':35356,'Unknown':7961}, 'Input/version mismatch: stop before exporting.'


## 06 — Group Split


In [ ]:
labels = ['Turbine_ID','Timestamp','Condition','Fault_Type','Binary_Eligible','Type_Eligible']
s = pd.read_csv(ROOT/'corrected_scada/scada_corrected_labels.csv.gz', usecols=labels+FEATURES)
s.Timestamp = pd.to_datetime(s.Timestamp, format='mixed')
s['Source_Row_ID'] = np.arange(len(s))
assert len(s) == 260367
assert not s.duplicated(['Turbine_ID','Timestamp']).any()
# Form conservative overlap groups from all recorded fault intervals AND
# consecutive fault-labelled SCADA runs. These are leakage-control groups,
# not independently verified physical failure episodes.
ev=pd.read_csv(ROOT/'corrected_scada/events_link_audit.csv.gz')
ev.Start=pd.to_datetime(ev.Start,format='mixed');ev.End=pd.to_datetime(ev.End,format='mixed')
ev=ev[(ev.Event_Class=='Fault')&ev.Time_Status.eq('Valid_Interval')]
s['Episode_ID']=''
episode_rows=[]
for turbine,idx0 in s.groupby('Turbine_ID').groups.items():
    idx=np.array(idx0);idx=idx[np.argsort(s.loc[idx,'Timestamp'].to_numpy())]
    f=s.loc[idx];f=f[f.Condition.eq('Fault')]
    intervals=[(r.Start,r.End-pd.Timedelta(nanoseconds=1)) for r in ev[ev.Turbine_ID.eq(turbine)].itertuples()]
    if len(f):
        run_id=f.Timestamp.diff().ne(pd.Timedelta(minutes=10)).cumsum()
        intervals.extend((g.Timestamp.min(),g.Timestamp.max()) for _,g in f.groupby(run_id))
    intervals.sort()
    merged=[]
    for start,end in intervals:
        if merged and start<=merged[-1][1]: merged[-1][1]=max(end,merged[-1][1])
        else: merged.append([start,end])
    times=s.loc[idx,'Timestamp'].to_numpy(dtype='datetime64[ns]')
    for j,(start,end) in enumerate(merged):
        a=np.searchsorted(times,np.datetime64(start,'ns'),'left')
        b=np.searchsorted(times,np.datetime64(end,'ns'),'right')
        hit=idx[a:b]
        hit=hit[s.loc[hit,'Condition'].eq('Fault').to_numpy()]
        if not len(hit):continue
        episode=f'{turbine}_E{j:04d}'
        s.loc[hit,'Episode_ID']=episode
assert s.loc[s.Condition.eq('Fault'),'Episode_ID'].ne('').all()

s['Group_ID'] = s.Episode_ID.fillna('')
for turbine, idx0 in s.groupby('Turbine_ID').groups.items():
    idx = np.array(idx0)
    idx = idx[np.argsort(s.loc[idx,'Timestamp'].to_numpy())]
    g = s.loc[idx]
    normal = g.Condition.eq('Normal')
    boundary = (~normal.shift(fill_value=False)) | g.Timestamp.diff().ne(pd.Timedelta(minutes=10))
    run = boundary.cumsum()
    s.loc[idx[normal.to_numpy()],'Group_ID'] = [f'{turbine}_N{v:05d}' for v in run[normal]]
eligible = s.Binary_Eligible
assert set(s.loc[eligible,'Group_ID']) == set(frozen.Group_ID), 'Group mismatch: stop.'
s['Split'] = s.Group_ID.map(frozen.set_index('Group_ID').Split).fillna('excluded_unknown')
assert s.loc[eligible].groupby('Group_ID').Split.nunique().eq(1).all()
actual = pd.crosstab(s.loc[eligible,'Group_ID'], s.loc[eligible,'Condition'])
expected = frozen.set_index('Group_ID')[['Normal','Fault']]
pd.testing.assert_frame_equal(actual.reindex(expected.index)[['Normal','Fault']], expected, check_names=False, check_dtype=False)
print(pd.crosstab(s.loc[eligible,'Split'], s.loc[eligible,'Condition']))


## 07 — Imputation & Scaling


In [ ]:
import joblib, sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MODEL_OUT = ROOT/'model_ready'
MODEL_OUT.mkdir(exist_ok=True)
selected = ['Electrical','Gear_Oil','Generator_Cooling','Lubrication','Pitch_Hydraulic','Valve_Signal']
X = s[FEATURES].replace([np.inf,-np.inf],np.nan)
train_mask = eligible & s.Split.eq('train')
assert not X.loc[train_mask].isna().all().any()
pipeline = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
pipeline.fit(X.loc[train_mask])
Z = pipeline.transform(X)
assert Z.shape == (260367,48) and np.isfinite(Z).all()
joblib.dump(pipeline, MODEL_OUT/'imputer_scaler_pipeline.joblib')
pd.DataFrame({'feature':FEATURES}).to_csv(MODEL_OUT/'features_final.csv',index=False)
frozen.to_csv(MODEL_OUT/'group_assignment.csv',index=False)
metadata = ['Source_Row_ID','Turbine_ID','Timestamp','Group_ID','Condition','Fault_Type']
expected_sizes = {'binary':[179855,36131,36420], 'fault_type':[23700,7445,2505]}
counts=[]
for j, split in enumerate(['train','validation','test']):
    for task in ['binary','fault_type']:
        mask = eligible & s.Split.eq(split)
        if task == 'fault_type':
            mask &= s.Type_Eligible & s.Fault_Type.isin(selected)
        ix = s.index[mask]
        out = s.loc[ix,metadata].copy()
        out = pd.concat([out,pd.DataFrame(Z[ix],index=ix,columns=FEATURES)],axis=1)
        out['Target'] = out.Condition.map({'Normal':0,'Fault':1}) if task=='binary' else out.Fault_Type
        assert len(out)==expected_sizes[task][j] and out.shape[1]==55
        out.to_csv(MODEL_OUT/f'{task}_{split}.csv.gz',index=False,compression={'method':'gzip','compresslevel':1})
        for name, count in out.Target.value_counts().items():
            counts.append(dict(Task=task,Split=split,Class=name,Rows=int(count)))
class_counts=pd.DataFrame(counts)
class_counts.to_csv(MODEL_OUT/'class_counts.csv',index=False)
s[metadata+['Split']].to_csv(MODEL_OUT/'row_assignment_audit.csv.gz',index=False)
summary={'master_rows':len(s),'binary_rows':int(eligible.sum()),'unknown_excluded':int((~eligible).sum()),
         'feature_count':48,'split':'Frozen delivered whole-group assignment',
         'selected_fault_types':selected,'sklearn_version':sklearn.__version__}
(MODEL_OUT/'summary.json').write_text(json.dumps(summary,indent=2))
print(class_counts.to_string(index=False))
print('Outputs:', MODEL_OUT)


## 08 — Export Excel (Optional)


In [ ]:
EXPORT_EXCEL = False
if EXPORT_EXCEL:
    import importlib.util, subprocess, sys
    if importlib.util.find_spec('xlsxwriter') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','xlsxwriter'])
    import xlsxwriter
    for split in ['train','validation','test']:
        destination = MODEL_OUT/f'SCADA_{split.title()}_All.xlsx'
        with xlsxwriter.Workbook(str(destination), {'constant_memory':True}) as book:
            sheet = book.add_worksheet('SCADA')
            row = 0
            for chunk in pd.read_csv(MODEL_OUT/f'binary_{split}.csv.gz',chunksize=5000):
                if row == 0:
                    sheet.write_row(row,0,chunk.columns.tolist());row += 1
                for values in chunk.itertuples(index=False,name=None):
                    sheet.write_row(row,0,values);row += 1
        print('Created:', destination.name)


## 09 — Download (Optional)


In [ ]:
FILE_TO_DOWNLOAD = MODEL_OUT/'class_counts.csv'
# Example: MODEL_OUT/'binary_train.csv.gz'
from google.colab import files
files.download(str(FILE_TO_DOWNLOAD))